# DAY 4

In [1]:
import os
import time
import getpass
import anthropic
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Paste your key into the box that appears (it is not saved in the code)
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your API key: ")

client = anthropic.Anthropic()          # created AFTER the key is set

test_300 = pd.read_csv("test_300.csv")
hard = pd.read_csv("hard_cases.csv")

def llm_label(text):
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=10,
        thinking={"type": "disabled"},      # ask the model not to "think" first
        messages=[{"role": "user", "content":
            "Is this review positive or negative? Reply with one word only.\n\n" + text}],
    )

    reply = ""
    for block in response.content:          # look through every piece of the reply
        if block.type == "text":            # keep only the actual answer text
            reply = block.text.lower()

    if "positive" in reply:
        label = "positive"
    elif "negative" in reply:
        label = "negative"
    else:
        label = "invalid"
    return label, response.usage.input_tokens, response.usage.output_tokens

# Try it on one review first
print(llm_label(test_300["text"].iloc[0]))

('negative', 157, 5)


In [2]:
import vaderSentiment, nltk, anthropic, sklearn

print("API key found:", os.environ.get("ANTHROPIC_API_KEY") is not None)
print("All packages imported")

API key found: True
All packages imported


In [3]:
tokens_in = 0
tokens_out = 0
preds = []

start = time.time()
for text in test_300["text"]:
    label, t_in, t_out = llm_label(text)
    preds.append(label)
    tokens_in += t_in
    tokens_out += t_out
seconds = time.time() - start

test_300["llm_pred"] = preds
test_300.to_csv("test_300_with_llm.csv", index=False)
                
hard["llm_pred"] = [llm_label(text)[0] for text in hard["text"]]

print("Done in", round(seconds), "seconds")
print(test_300.columns.tolist())
print("Invalid replies:", (test_300["llm_pred"] == "invalid").sum())

Done in 235 seconds
['label', 'text', 'y', 'llm_pred']
Invalid replies: 10


In [4]:
cost = (tokens_in * 0.10 + tokens_out * 0.50) / 1_000_000    # dollars for 300 reviews

accuracy = accuracy_score(test_300["y"], test_300["llm_pred"])
macro_f1 = f1_score(test_300["y"], test_300["llm_pred"], average="macro", labels=["negative", "positive"])
hard_accuracy = accuracy_score(hard["y"], hard["llm_pred"])
matrix = confusion_matrix(test_300["y"], test_300["llm_pred"], labels=["negative", "positive"])

print("Accuracy:", round(accuracy, 3))
print("Macro-F1:", round(macro_f1, 3))
print("Hard-cases accuracy:", round(hard_accuracy, 3))
print("Seconds per 1,000:", round(seconds / 300 * 1000, 1))
print("Cost per 1,000: $", round(cost / 300 * 1000, 4))
print("Invalid replies:", (test_300["llm_pred"] == "invalid").sum())
print(matrix)

pd.DataFrame([{
    "Model": "LLM zero-shot",
    "Accuracy": round(accuracy, 3),
    "Macro-F1": round(macro_f1, 3),
    "Hard cases acc.": round(hard_accuracy, 3),
    "Sec per 1k preds": round(seconds / 300 * 1000, 1),
    "Cost per 1k preds": round(cost / 300 * 1000, 4),
}]).to_csv("results_llm.csv", index=False)

Accuracy: 0.96
Macro-F1: 0.976
Hard-cases accuracy: 0.933
Seconds per 1,000: 783.4
Cost per 1,000: $ 0.0282
Invalid replies: 10
[[146   1]
 [  1 142]]
